# 12. LR vs RN — Séparation du bloc Droite

Ce notebook analyse la distinction entre Les Républicains (LR) et Rassemblement National (RN) dans le bloc Droite (tâche G2). Effectifs par batch, stance mensuel, fighting words, Mann-Whitney, zoom sur CEASEFIRE_BREACH.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))
import re
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

from config import (
    CORPUS_V3, BLOC_ORDER_5, BLOC_COLORS_5, BATCHES,
    add_events, format_dates, month_to_batch, add_bloc_5
)

FIG_DIR = Path("../figures")
RES_DIR = Path("../data/results")
FIG_DIR.mkdir(exist_ok=True)

def tokenize(t):
    return re.findall(r"[a-zàâäéèêëïîôùûüç]+", str(t).lower())

def log_odds(cnt1, cnt2, alpha=0.01):
    n1 = sum(cnt1.values()) + alpha * 2
    n2 = sum(cnt2.values()) + alpha * 2
    out = {}
    for w in set(cnt1) | set(cnt2):
        c1 = cnt1.get(w, 0) + alpha
        c2 = cnt2.get(w, 0) + alpha
        delta = np.log(c1 / n1) - np.log(c2 / n2)
        var = 1 / c1 + 1 / c2
        out[w] = delta / np.sqrt(var) if var > 0 else 0
    return out

## Chargement et effectifs

In [ ]:
df = pd.read_parquet(CORPUS_V3)
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.to_period("M").astype(str)
df["batch"] = df["month"].apply(month_to_batch)
df_5 = add_bloc_5(df)

eff = df_5.groupby(["batch", "bloc_5"]).size().unstack(fill_value=0)
print("Effectifs par batch × bloc_5 :")
print(eff.to_string())
print("\nLR vs RN (Droite) :")
droite = df[df["bloc"] == "Droite"]
print(droite["group"].value_counts())

## Stance mensuel par bloc_5 (5 courbes)

In [ ]:
stance_m = df_5.groupby(["month", "bloc_5"])["stance_v3"].mean().reset_index()
stance_m["month_ts"] = pd.to_datetime(stance_m["month"] + "-01")
stance_m.to_csv(RES_DIR / "stance_mensuel_bloc5.csv", index=False)

fig, ax = plt.subplots(figsize=(14, 6))
for bloc in BLOC_ORDER_5:
    sub = stance_m[stance_m["bloc_5"] == bloc]
    if len(sub) > 0:
        ax.plot(sub["month_ts"], sub["stance_v3"], label=bloc, color=BLOC_COLORS_5.get(bloc, "#888"), lw=2)
add_events(ax)
ax.axhline(0, color="grey", ls="--", lw=0.8)
ax.legend(loc="upper right")
ax.set_ylabel("Positionnement moyen")
ax.set_title("Stance mensuel : LR vs RN séparés")
plt.savefig(FIG_DIR / "fig_lr_rn_stance_ribbon.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

## Fighting words LR vs RN

In [ ]:
tc = "text_clean" if "text_clean" in df_5.columns else "text"
lr = df_5[df_5["bloc_5"] == "Droite LR"][tc].dropna()
rn = df_5[df_5["bloc_5"] == "Droite RN"][tc].dropna()
cnt_lr = Counter()
cnt_rn = Counter()
for t in lr: cnt_lr.update(tokenize(t))
for t in rn: cnt_rn.update(tokenize(t))

lodds = log_odds(cnt_lr, cnt_rn)
sorted_w = sorted(lodds.items(), key=lambda x: -abs(x[1]))[:25]
words = [w for w, _ in sorted_w]
vals = [v for _, v in sorted_w]

fig, ax = plt.subplots(figsize=(10, 8))
colors = ["#3498db" if v > 0 else "#2c3e50" for v in vals]
ax.barh(words, vals, color=colors)
ax.axvline(0, color="grey", ls="--")
ax.set_xlabel("Log-odds (LR vs RN) — positif = sur-représenté chez LR")
ax.set_title("Fighting words : Droite LR vs Droite RN")
plt.savefig(FIG_DIR / "fig_lr_rn_fighting_words.png", dpi=150, bbox_inches="tight", facecolor="white")
plt.show()

## Mann-Whitney LR vs RN par batch

In [ ]:
mw_res = []
for batch in df_5["batch"].dropna().unique():
    lr_s = df_5[(df_5["bloc_5"] == "Droite LR") & (df_5["batch"] == batch)]["stance_v3"]
    rn_s = df_5[(df_5["bloc_5"] == "Droite RN") & (df_5["batch"] == batch)]["stance_v3"]
    if len(lr_s) >= 5 and len(rn_s) >= 5:
        stat, p = mannwhitneyu(lr_s, rn_s, alternative="two-sided")
        mw_res.append({"batch": batch, "lr_mean": lr_s.mean(), "rn_mean": rn_s.mean(), "delta": lr_s.mean() - rn_s.mean(), "p": p})
mw_df = pd.DataFrame(mw_res)
mw_df.to_csv(RES_DIR / "mann_whitney_lr_rn.csv", index=False)
print(mw_df.to_string(index=False))

## Zoom CEASEFIRE_BREACH — Paradoxe Droite LR vs RN ?

In [ ]:
cfb = df_5[df_5["batch"] == "CEASEFIRE_BREACH"]
if len(cfb) > 0:
    sm = cfb.groupby("bloc_5")["stance_v3"].agg(["mean", "count"])
    print("Stance moyen CEASEFIRE_BREACH :")
    print(sm.to_string())
    lr_cfb = cfb[cfb["bloc_5"] == "Droite LR"]["stance_v3"]
    rn_cfb = cfb[cfb["bloc_5"] == "Droite RN"]["stance_v3"]
    if len(lr_cfb) >= 3 and len(rn_cfb) >= 3:
        _, p = mannwhitneyu(lr_cfb, rn_cfb, alternative="two-sided")
        print(f"\nMann-Whitney LR vs RN (CEASEFIRE_BREACH) : p = {p:.4f}")
        print("Le paradoxe Droite au cessez-le-feu est-il drivé par LR, RN ou les deux ?")
        print("→ Comparer les deltas par rapport aux batches précédents.")